<a href="https://colab.research.google.com/github/williamlidberg/Unet-tutorial/blob/main/Introduction_to_object_detection_with_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to object detection using YOLO
Make sure to change runtype to GPU before you start. If you have multiple GPU options just pick one at random.

# 1. Load Data
We download the pre-prepared YOLO dataset directly from Williams Google Drive and unpack it.

In [ ]:
!pip install ultralytics


# Download and unpack dataset


In [ ]:
import os
import shutil
# This downloads the data
if not os.path.exists('/content/yolo_dataset.zip'):
    print("Downloading dataset...")
    !gdown --id 1jGPBj8ToTKhMMMsdI4qy0cE65CKT3FZS -O /content/yolo_dataset.zip
# This unzips it
if not os.path.exists('/content/yolo_dataset'):
    print("Extracting dataset...")
    shutil.unpack_archive('/content/yolo_dataset.zip', '/content/')
    print("Extraction complete!")

# 2. Plot Examples of Data
We select random samples from our dataset images to inspect them.

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

# Updated directory paths to use /content directly
train_images_dir = '/content/images/train'
sample_files = random.sample(os.listdir(train_images_dir), 3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, file_name in enumerate(sample_files):
    img_path = os.path.join(train_images_dir, file_name)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(file_name)
    axes[i].axis('off')
plt.show()

# 3. Plot Examples with YAML Ground Truth Labels
We visualize a training image alongside its bounding box annotations as specified by our `dataset.yaml` definition.

In [ ]:
import yaml
import os
import random
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Create dataset.yaml config (Updated path to point directly to /content)
yolo_config = {
    'path': '/content',
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'hunting_pit'}
}
with open('/content/dataset.yaml', 'w') as f:
    yaml.dump(yolo_config, f, default_flow_style=False)

def visualize_yolo_sample(images_dir, labels_dir):
    label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
    if not label_files:
        print("No label files found."); return

    # Pick a random sample with labels
    random.shuffle(label_files)
    for lf in label_files:
        with open(os.path.join(labels_dir, lf)) as f:
            boxes = [line.strip().split() for line in f.readlines() if line.strip()]
        if boxes:
            img_path = os.path.join(images_dir, lf.replace('.txt', '.png'))
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            h, w = img.shape
            fig, ax = plt.subplots(1, figsize=(8, 8))
            ax.imshow(img, cmap='gray')
            for box in boxes:
                cid, x_c, y_c, bw, bh = map(float, box)
                rect = patches.Rectangle(
                    ((x_c - bw/2)*w, (y_c - bh/2)*h), bw*w, bh*h,
                    linewidth=2, edgecolor='red', facecolor='none'
                )
                ax.add_patch(rect)
            plt.title(f"Ground Truth: {lf.replace('.txt', '.png')}")
            plt.axis('off')
            plt.show()
            break

visualize_yolo_sample('/content/images/train', '/content/labels/train')

# 4. Augment Data
Configure YOLO real-time data augmentations such as scale, translation, and horizontal flip during training.

In [ ]:
# Setup YOLO training parameters with built-in augmentations enabled
augmentation_params = {
    'degrees': 15.0,
    'translate': 0.1,
    'scale': 0.5,
    'fliplr': 0.5
}
print("Augmentations configured successfully!")

# 5. Train YOLO Model
Start training the model with optimized hyperparameters and minimal validation metrics output.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8x.pt')
results = model.train(
    data='/content/dataset.yaml',
    epochs=5,
    imgsz=256,
    batch=16,
    device=0,
    **augmentation_params # Change augmentation parameters above and see what
)

# 6. Show Train and Loss Curves
Visualize how the loss metrics evolved during model optimization epochs. If the lines dont flatten out you might want to train the model for longer. Change epochs=5 to something larger and run again.

In [ ]:
import os
from IPython.display import display, Image as IPImage

# Read generated training results image
latest_train_dir = sorted([d for d in os.listdir('/content/runs/detect') if d.startswith('train')])[-1]
results_img_path = os.path.join('/content/runs/detect', latest_train_dir, 'results.png')

if os.path.exists(results_img_path):
    display(IPImage(filename=results_img_path))
else:
    print("Results plot not found.")

# 7. Run Model on All Test Data & Show Final Metrics
We evaluate our optimized model against the complete validation partition.

In [ ]:
# Run full evaluation loop
metrics = model.val(data='/content/dataset.yaml', split='val', imgsz=256, device=0)

precision = metrics.results_dict['metrics/precision(B)']
recall = metrics.results_dict['metrics/recall(B)']
map50 = metrics.results_dict['metrics/mAP50(B)']

# Calculate overall F1-Score mathematically
if (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
else:
    f1_score = 0.0

print("\n--- Final Model Metrics (F1 and mAP) ---")
print(f"F1-Score (All classes): {f1_score:.4f}")
print(f"mAP50: {map50:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Precision: This is your model's reliability check. It measures how often the model is correct when it claims to have found a hunting_pit. High precision means that when a box is drawn, it is highly likely to be a real target rather than a false alarm.\
\
Recall: This is your model's search thoroughness. It measures the percentage of all actual, existing hunting_pits in the dataset that the model successfully located. High recall means the model is good at finding targets and doesn't leave many behind.\
\
F1-Score (The Balanced score): This is the harmonious middle ground between Precision and Recall. It gives you a single baseline showing how well your model balances finding as many targets as possible (Recall) while maintaining a high rate of accuracy for each detection (Precision).\
\
mAP50 (The Boundary Quality): This evaluates both class identification and bounding box accuracy. It measures how cleanly the model's predicted boxes align and overlap with the actual shapes of the targets. A strong score here indicates the model is highly capable of precisely framing and mapping the targets.

# 8. Run Model on 6 Images and Plot as a Grid
We run inference on six random validation images and plot the predicted bounding boxes together.

In [ ]:
import random
import cv2
import os
import matplotlib.pyplot as plt

# Updated directory paths to use /content directly
val_images_dir = '/content/images/val'
val_files = [f for f in os.listdir(val_images_dir) if f.endswith('.png') or f.endswith('.tif')]
sample_6 = random.sample(val_files, 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, img_name in enumerate(sample_6):
    img_path = os.path.join(val_images_dir, img_name)
    results = model.predict(source=img_path, conf=0.25, verbose=False)

    ax = axes[i // 3, i % 3]
    if len(results) > 0:
        im_array = results[0].plot()
        im_rgb = cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB)
        ax.imshow(im_rgb)
    else:
        img_orig = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        ax.imshow(img_orig, cmap='gray')

    ax.set_title(f"Inference: {img_name}")
    ax.axis('off')

plt.tight_layout()
plt.show()

When you trained the model you used yolov8n.pt (Nano): The default model. It has the smallest file size, uses the least amount of GPU memory, and trains/infers the fastest. It is ideal for quick testing or edge devices. However, there are other models, larger and more capable models that you can try. Attempt to train one of the following:\
\
yolov8s.pt (Small): Slightly larger than Nano. It offers a noticeable improvement in accuracy while still being very fast and lightweight.\
\
yolov8m.pt (Medium): A well-balanced model scale. It is a great default choice for datasets of moderate complexity when you want solid accuracy without sacrificing too much training speed.\
\
yolov8l.pt (Large): A heavy-duty model. It is designed to capture complex details and features, though it requires more GPU memory and takes longer to train.\
\
yolov8x.pt (Extra-Large): The most accurate and complex model in the standard YOLOv8 family. It has the highest boundary mapping precision but is also the most resource-intensive.\
\
scroll back up to this cell: model = YOLO('yolov8s.pt') and change the model name. then Train the new model